In [1]:
"""
inspect_units_fc3.py
Inspecciona las unidades disponibles en dev y test para Fc=3,
replicando exactamente el split del notebook XGBoost_N-CMAPSS_Fc3.ipynb.
"""

import h5py
import numpy as np
import pandas as pd

FILE_PATH = 'N-CMAPSS_DS01-005.h5'
FC        = 3

# ── Carga ──────────────────────────────────────────────────────────────────
with h5py.File(FILE_PATH, 'r') as f:
    W_dev    = np.array(f['W_dev'])
    X_s_dev  = np.array(f['X_s_dev'])
    Y_dev    = np.array(f['Y_dev'])
    A_dev    = np.array(f['A_dev'])
    W_test   = np.array(f['W_test'])
    X_s_test = np.array(f['X_s_test'])
    Y_test   = np.array(f['Y_test'])
    A_test   = np.array(f['A_test'])
    W_var    = [v.decode() for v in f['W_var'][:]]
    Xs_var   = [v.decode() for v in f['X_s_var'][:]]
    A_var    = [v.decode() for v in f['A_var'][:]]

# ── Excluir hs ─────────────────────────────────────────────────────────────
hs_idx    = A_var.index('hs') if 'hs' in A_var else None
A_var_use = [v for v in A_var if v != 'hs']
A_dev_use  = np.delete(A_dev,  hs_idx, axis=1) if hs_idx is not None else A_dev
A_test_use = np.delete(A_test, hs_idx, axis=1) if hs_idx is not None else A_test

all_cols = A_var_use + W_var + Xs_var + ['RUL']

df_dev_full = pd.DataFrame(
    np.hstack([A_dev_use, W_dev, X_s_dev, Y_dev.reshape(-1, 1)]),
    columns=all_cols
)
df_test_full = pd.DataFrame(
    np.hstack([A_test_use, W_test, X_s_test, Y_test.reshape(-1, 1)]),
    columns=all_cols
)

for col in ['unit', 'cycle', 'Fc']:
    df_dev_full[col]  = df_dev_full[col].astype(int)
    df_test_full[col] = df_test_full[col].astype(int)

# ── Filtro Fc=3 ────────────────────────────────────────────────────────────
df_dev_fc3  = df_dev_full[df_dev_full['Fc'] == FC].copy()
df_test_fc3 = df_test_full[df_test_full['Fc'] == FC].copy()

# ── Split por unidad (igual que Cell 7 del notebook) ──────────────────────
units_sorted = sorted(df_dev_fc3['unit'].unique())
val_unit     = [units_sorted[-1]]
train_units  = units_sorted[:-1]
units_test   = sorted(df_test_fc3['unit'].unique())

# ── Reporte ────────────────────────────────────────────────────────────────
print("=" * 60)
print(f"UNIDADES DISPONIBLES — Fc={FC}  |  DS01-005")
print("=" * 60)

print(f"\nDEV total   : {len(units_sorted)} unidades → {units_sorted}")
print(f"  TRAIN     : {len(train_units)} unidades → {train_units}")
print(f"  VALIDATION: {len(val_unit)} unidad    → {val_unit}")
print(f"\nTEST        : {len(units_test)} unidades → {units_test}")

print("\n" + "-" * 60)
print("Observaciones y ciclos por split:")
print("-" * 60)

for label, units, df in [
    ('TRAIN', train_units, df_dev_fc3),
    ('VAL',   val_unit,    df_dev_fc3),
    ('TEST',  units_test,  df_test_fc3),
]:
    sub = df[df['unit'].isin(units)]
    for u in units:
        u_df = sub[sub['unit'] == u]
        print(f"  [{label}] Unidad {u:3d} — "
              f"ciclos: {u_df['cycle'].nunique():4d}  |  "
              f"obs: {len(u_df):7,}  |  "
              f"RUL [{u_df['RUL'].min():.0f}, {u_df['RUL'].max():.0f}]")

UNIDADES DISPONIBLES — Fc=3  |  DS01-005

DEV total   : 2 unidades → [2, 5]
  TRAIN     : 1 unidades → [2]
  VALIDATION: 1 unidad    → [5]

TEST        : 1 unidades → [10]

------------------------------------------------------------
Observaciones y ciclos por split:
------------------------------------------------------------
  [TRAIN] Unidad   2 — ciclos:   75  |  obs: 1,049,088  |  RUL [0, 74]
  [VAL] Unidad   5 — ciclos:   89  |  obs: 1,280,147  |  RUL [0, 88]
  [TEST] Unidad  10 — ciclos:   82  |  obs: 1,190,670  |  RUL [0, 81]
